\begin{center}
{\LARGE \textbf{EN3160 Image Processing and Machine Vision}}\\[0.5em]
{\Large Assignment 2 -- Fitting and Alignment}\\[1.5em]

\textbf{Name:} S.A.D. Priyadarshana\\[0.3em]
\textbf{Index:} 230502X\\[0.3em]
\textbf{GitHub:} \href{https://github.com/denethp/CV-Assignment-02}{denethp/CV-Assignment-02}
\end{center}

\vspace{1em}

$\underline{\textbf{Question 1}}$

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter

%matplotlib inline

In [ ]:
def log_kernel(sigma):
    # Scale-normalized LoG, window of +-3 sigma;
    # mean removed so the truncated kernel gives zero response on flat regions.
    hw = int(np.ceil(3 * sigma))
    X, Y = np.meshgrid(np.arange(-hw, hw + 1), np.arange(-hw, hw + 1))
    log = 1 / (2 * np.pi * sigma ** 2) * (X ** 2 / sigma ** 2 + Y ** 2 / sigma ** 2 - 2) \
          * np.exp(-(X ** 2 + Y ** 2) / (2 * sigma ** 2))
    return log - log.mean()

In [ ]:
im = cv.imread('images/the_berry_farms_sunflower_field.jpeg', cv.IMREAD_REDUCED_COLOR_4)
gray = cv.cvtColor(im, cv.COLOR_BGR2GRAY).astype(np.float32) / 255.0

sigmas = np.arange(1, 16.5, 0.5)
scale_space = np.stack([cv.filter2D(gray, -1, log_kernel(s)) for s in sigmas], axis=2)

# Scale-space extrema: maximum in the 3x3x3 (x, y, sigma) neighbourhood above a threshold.
# Dark sunflower centres on bright petals give positive LoG peaks.
threshold = 0.2
peaks = (scale_space == maximum_filter(scale_space, size=3)) & (scale_space > threshold)
peaks[:, :, [0, -1]] = False  # end scales have only one neighbouring scale

ys, xs, ks = np.nonzero(peaks)
radii = np.sqrt(2) * sigmas[ks]  # r = sqrt(2) * sigma
print(f'sigma range: {sigmas[0]} to {sigmas[-1]} (step 0.5), blobs detected: {len(ys)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.3, 4.6))
axes[0].imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
for x, y, r in zip(xs, ys, radii):
    axes[1].add_patch(plt.Circle((x, y), r, color='r', fill=False, linewidth=1))
axes[1].set_title('Detected circles (LoG)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
order = np.argsort(-radii)[:5]
print('  x    y   sigma  radius')
for o in order:
    print(f'{xs[o]:3d}  {ys[o]:3d}  {sigmas[ks[o]]:5.1f}  {radii[o]:6.2f}')

$\textbf{Discussion:}$ the image is converted to grayscale and filtered with scale-normalized LoG kernels for $\sigma \in [1, 16]$ (step 0.5). A blob is a point that is a maximum among its 26 neighbours in $(x, y, \sigma)$ and whose response exceeds 0.2; the circle radius is $r = \sqrt{2}\sigma$. The largest circles are at (338, 165) with $\sigma = 13$ ($r \approx 18.4$) and (293, 168) with $\sigma = 12$ ($r \approx 17.0$), but these lie on the dark tree line against the bright sky rather than on flowers. The largest circles on sunflowers have $\sigma = 7.5$ ($r \approx 10.6$), e.g. the large flower at (282, 338). Almost every sunflower centre is detected, and smaller $\sigma$ values pick up the more distant flowers, showing the scale selection at work. Because LoG responds to any dark blob on a lighter background, some false detections appear along the tree line and in leaf shadows; a higher threshold or restricting detection to the field region would remove these.